# Leer Data

**John González**

*23 de mayo de 2026*

**Objetivo**: Leer y guardar la data

In [ ]:
# ! uv add kagglehub

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("netflix-inc/netflix-prize-data")

print("Path to dataset files:", path)

/workspaces/bigdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/vscode/.cache/kagglehub/datasets/netflix-inc/netflix-prize-data/versions/2


In [3]:
# Leer algunas lineas de path/combined_data_1.txt
NUM_LINES_TO_READ = 20

file_path = f"{path}/combined_data_1.txt"

with open(file_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f, start=1):
        print(line.rstrip())
        if i >= NUM_LINES_TO_READ:
            break

1:
1488844,3,2005-09-06
822109,5,2005-05-13
885013,4,2005-10-19
30878,4,2005-12-26
823519,3,2004-05-03
893988,3,2005-11-17
124105,4,2004-08-05
1248029,3,2004-04-22
1842128,4,2004-05-09
2238063,3,2005-05-11
1503895,4,2005-05-19
2207774,5,2005-06-06
2590061,3,2004-08-12
2442,3,2004-04-14
543865,4,2004-05-28
1209119,4,2004-03-23
804919,4,2004-06-10
1086807,3,2004-12-28
1711859,4,2005-05-08


In [1]:
# Leer algunas lineas de path/combined_data_1.txt
NUM_LINES_TO_READ = 20

file_path = f"{path}/movie_titles.csv"

with open(file_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f, start=1):
        print(line.rstrip())
        if i >= NUM_LINES_TO_READ:
            break

NameError: name 'path' is not defined

In [9]:
# Leer algunas lineas de path/combined_data_1.txt
NUM_LINES_TO_READ = 20

file_path = f"{path}/qualifying.txt"

with open(file_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f, start=1):
        print(line.rstrip())
        if i >= NUM_LINES_TO_READ:
            break

1:
1046323,2005-12-19
1080030,2005-12-23
1830096,2005-03-14
368059,2005-05-26
802003,2005-11-07
513509,2005-07-04
1086137,2005-09-21
428698,2005-12-20
515850,2005-11-27
131974,2005-12-15
1414572,2005-12-19
2112862,2004-12-02
1986275,2005-01-22
2168239,2005-11-26
441146,2005-05-25
1687310,2005-09-30
1635114,2005-06-06
2455305,2005-10-07
645660,2005-12-21


In [8]:
# Leer algunas lineas de path/combined_data_1.txt
NUM_LINES_TO_READ = 20

file_path = f"{path}/probe.txt"

with open(file_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f, start=1):
        print(line.rstrip())
        if i >= NUM_LINES_TO_READ:
            break

1:
30878
2647871
1283744
2488120
317050
1904905
1989766
14756
1027056
1149588
1394012
1406595
2529547
1682104
2625019
2603381
1774623
470861
712610


## Organizar la tabla combined_data_1

Se organiza la tabla combined_data_1 para que pase del formato:
movie_id_1:
CustomerID,Rating,Date
CustomerID,Rating,Date
CustomerID,Rating,Date
...
movie_id_2:
CustomerID,Rating,Date
CustomerID,Rating,Date
CustomerID,Rating,Date
...

tenga el siguiente formato

movie_id_1,CustomerID,Rating,Date
movie_id_1,CustomerID,Rating,Date
movie_id_1,CustomerID,Rating,Date
movie_id_1,CustomerID,Rating,Date
movie_id_2,CustomerID,Rating,Date
movie_id_2,CustomerID,Rating,Date
movie_id_2,CustomerID,Rating,Date
movie_id_2,CustomerID,Rating,Date

### Convertir combined_data_1 con Spark

Leer el fichero como texto, extraer `movie_id`, parsear cada reseña y guardar en `parquet` para consultas y modelado de recomendación.

In [3]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder \
    .appName("NetflixPrizeRead") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

combined_path = f"{path}/combined_data_1.txt"

path_proyecto = "/workspaces/bigdata/proyecto/datos"
os.makedirs(path_proyecto, exist_ok=True)
output_path = f"{path_proyecto}/combined_data_1.parquet"

def parse_partition(partition):
    current_movie = None
    for line in partition:
        line = line.strip()
        if not line:
            continue
        if line.endswith(":"):
            try:
                current_movie = int(line[:-1])
            except ValueError:
                current_movie = None
            continue
        if current_movie is None:
            continue
        parts = line.split(",", 2)
        if len(parts) != 3:
            continue
        customer_id, rating, date = parts
        try:
            yield (
                current_movie,
                int(customer_id),
                int(rating),
                date.strip(),
            )
        except ValueError:
            continue

rdd = spark.sparkContext.textFile(combined_path)
parsed_rdd = rdd.mapPartitions(parse_partition)

schema = StructType([
    StructField("movie_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=False),
    StructField("rating", IntegerType(), nullable=False),
    StructField("date", StringType(), nullable=True),
])

review_df = spark.createDataFrame(parsed_rdd, schema)
review_df.printSchema()
print("Rows parsed:", review_df.count())

review_df.write.mode("overwrite").parquet(output_path)
print(f"Saved parquet to: {output_path}")


root
 |-- movie_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- rating: integer (nullable = false)
 |-- date: string (nullable = true)



Rows parsed: 23481434


26/05/23 19:19:06 WARN BasicWriteTaskStatsTracker: Expected 1 files, but only saw 0. This could be due to the output format not writing empty files, or files being not immediately visible in the filesystem.
26/05/23 19:19:06 WARN BasicWriteTaskStatsTracker: Expected 1 files, but only saw 0. This could be due to the output format not writing empty files, or files being not immediately visible in the filesystem.


Saved parquet to: /workspaces/bigdata/proyecto/datos/combined_data_1.parquet
